#### Preparation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import swin_t, Swin_T_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

In [2]:
DATA_DIR = r'C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification'
WEIGHTS_DIR = r'../weights'

BATCH_SIZE = 32
FREEZE_EPOCHS = 8
FINETUNE_EPOCHS = 20

LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
weights = Swin_T_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

In [4]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)

Classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [5]:
model = swin_t(weights=weights)

In [6]:
for param in model.parameters():
    param.requires_grad = False

In [7]:
# Replace Classification Head
in_features = model.head.in_features

model.head = nn.Linear(in_features, num_classes)

In [8]:
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.head.parameters(),
    lr=LR
)

In [9]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss, train_acc


In [10]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating "):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss, val_acc

#### Training 1 (Freeze-Backbone)

In [11]:
for epoch in range(FREEZE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FREEZE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FREEZE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")

Validating : 100%|██████████| 8/8 [00:45<00:00,  5.64s/it]



Epoch [1/8]
Train Loss: 29.6437 | Train Acc: 0.6375
Val Loss: 5.5145 | Val Acc: 0.7292
Precision: 0.7579 | Recall: 0.7292 | F1: 0.7165




Validating : 100%|██████████| 8/8 [00:42<00:00,  5.28s/it]



Epoch [2/8]
Train Loss: 18.0586 | Train Acc: 0.8187
Val Loss: 4.3301 | Val Acc: 0.7792
Precision: 0.7990 | Recall: 0.7792 | F1: 0.7735




Validating : 100%|██████████| 8/8 [00:43<00:00,  5.41s/it]


...


Epoch [6/8]
Train Loss: 11.1095 | Train Acc: 0.8792
Val Loss: 2.8994 | Val Acc: 0.8708
Precision: 0.8798 | Recall: 0.8708 | F1: 0.8697




Validating : 100%|██████████| 8/8 [00:43<00:00,  5.38s/it]



Epoch [7/8]
Train Loss: 10.0908 | Train Acc: 0.9021
Val Loss: 2.6692 | Val Acc: 0.8917
Precision: 0.8973 | Recall: 0.8917 | F1: 0.8912




Validating : 100%|██████████| 8/8 [00:43<00:00,  5.43s/it]


Epoch [8/8]
Train Loss: 9.8191 | Train Acc: 0.8948
Val Loss: 2.6795 | Val Acc: 0.8708
Precision: 0.8798 | Recall: 0.8708 | F1: 0.8697




#### Training 2 (Fine-Tuning)

In [12]:
# Unfreeze last block and head for fine-tuning
for param in model.features[-1].parameters():
    param.requires_grad = True

In [13]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

In [14]:
best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(FINETUNE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FINETUNE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FINETUNE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names
        }, os.path.join(WEIGHTS_DIR, "Swint-T.pth"))


-----------Starting Fine-tuning Stage-----------



Validating : 100%|██████████| 8/8 [00:44<00:00,  5.58s/it]



Epoch [1/20]
Train Loss: 7.9680 | Train Acc: 0.9104
Val Loss: 2.2841 | Val Acc: 0.9208
Precision: 0.9284 | Recall: 0.9208 | F1: 0.9222




Validating : 100%|██████████| 8/8 [00:45<00:00,  5.68s/it]



Epoch [2/20]
Train Loss: 6.0258 | Train Acc: 0.9260
Val Loss: 1.9520 | Val Acc: 0.9167
Precision: 0.9207 | Recall: 0.9167 | F1: 0.9172




...


Epoch [18/20]
Train Loss: 1.3450 | Train Acc: 0.9854
Val Loss: 1.3349 | Val Acc: 0.9583
Precision: 0.9610 | Recall: 0.9583 | F1: 0.9588




Validating : 100%|██████████| 8/8 [00:44<00:00,  5.52s/it]



Epoch [19/20]
Train Loss: 0.9318 | Train Acc: 0.9896
Val Loss: 1.1502 | Val Acc: 0.9500
Precision: 0.9512 | Recall: 0.9500 | F1: 0.9503




Validating : 100%|██████████| 8/8 [00:44<00:00,  5.51s/it]


Epoch [20/20]
Train Loss: 1.2070 | Train Acc: 0.9833
Val Loss: 1.5626 | Val Acc: 0.9458
Precision: 0.9486 | Recall: 0.9458 | F1: 0.9464




#### Prediction Sample

In [16]:
checkpoint = torch.load(r"../weights/Swint-T.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]

# Example usage:
prediction = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Healthy\Healthy_val_13.jpg")
prediction1 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Blight\leaf blight_val_15.jpg")
prediction2 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Spot\leaf spot_val_14.jpg")
prediction3 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Spider Mites\Spider Mites_val_24.jpg")
print(f"Healthy Predicted class: {prediction}")
print(f"Leaf Blight Predicted class: {prediction1}")
print(f"Leaf Spot Predicted class: {prediction2}")
print(f"Spider Mites Predicted class: {prediction3}")

Healthy Predicted class: Healthy
Leaf Blight Predicted class: leaf blight
Leaf Spot Predicted class: Spider Mites
Spider Mites Predicted class: Spider Mites
